# 🍏 Fun & Fit Health Advisor Agent Tutorial 🍎

Welcome to our **Fun & Fit Health Advisor Agent** tutorial, where you'll use **Azure AI Foundry** SDKs to create a playful (yet carefully disclaimed!) health and fitness assistant. We'll:

1. **Initialize** our project using **azure-ai-projects**.
2. **Create an Agent** specialized in providing general wellness and nutritional advice (with disclaimers!).
3. **Manage conversations** about fitness, nutrition, and general health topics.
4. **Showcase logging and tracing** with **OpenTelemetry**.
5. **Demonstrate** how to incorporate tools, safety disclaimers, and basic best practices.

### ⚠️ Important Medical Disclaimer ⚠️
> **The health information provided by this notebook is for general educational and entertainment purposes only and is not intended as a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Never disregard professional medical advice or delay seeking it because of something you read or receive from this notebook.


## Prerequisites

Complete [the notebooks in introduction](../../1-introduction/3-quick_start.ipynb)

## Let's Get Started
We'll walk you through each cell with notes and diagrams to keep it fun. Let's begin!

<img src="./seq-diagrams/1-basics.png" width="30%"/>




## 1. Initial Setup
We'll start by importing needed libraries, loading environment variables, and initializing an **AIProjectClient** so we can do all the agent-related actions. Let's do it! 🎉


In [ ]:
import os
import time
import uuid
import requests
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import List
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MessageTextContent
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    # List all accessible subscriptions
    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None

    for sub in subs:
        sub_id = sub["subscriptionId"]
        # Search for the Foundry hub
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect subscription and resource group: {e}") from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"❌ Error initializing project client: {str(e)}")

# Initialize ChatCompletionsClient for Azure AI Foundry models (Phi-4, DeepSeek, etc.)
# These models use the /models endpoint, NOT /openai/v1
chat_endpoint = f"{base_endpoint}/models"
chat_client = ChatCompletionsClient(
    endpoint=chat_endpoint,
    credential=AzureKeyCredential(os.getenv("AZURE_OPENAI_KEY"))
)
print(f"✅ ChatCompletionsClient initialized | Endpoint: {chat_endpoint}")

## 2. Creating our Fun & Fit Health Advisor Agent 🏋️

We'll create an Agent specialized in general health and wellness. We'll explicitly mention disclaimers in its instructions, so it never forgets to keep it safe! The instructions also ask the agent to focus on general fitness, dietary tips, and always encourage the user to seek professional advice.


In [ ]:
# A tiny data structure that mimics the shape of the old server-side "agent" object,
# so the rest of the notebook keeps the same feel: "agent.id", "agent.name", etc.
@dataclass
class LocalAgent:
    id: str
    name: str
    model: str
    instructions: str


def create_health_advisor_agent():
    """Create a client-side 'health advisor' agent (system prompt + model deployment)."""
    try:
        # Get the model deployment name from the .env, default to a common one if missing
        model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")

        agent = LocalAgent(
            id=f"agent_{uuid.uuid4().hex[:8]}",
            name="fun-fit-health-advisor",
            model=model_name,
            instructions=(
                "You are a friendly AI Health Advisor.\n"
                "You provide general health, fitness, and nutrition information, but always:\n"
                "1. Include medical disclaimers.\n"
                "2. Encourage the user to consult healthcare professionals.\n"
                "3. Provide general, non-diagnostic advice around wellness, diet, and fitness.\n"
                "4. Clearly remind them you're not a doctor.\n"
                "5. Encourage safe and balanced approaches to exercise and nutrition."
            ),
        )
        # Log success and return the created agent
        print(f"🎉 Created health advisor agent, ID: {agent.id}")
        return agent

    except Exception as e:
        # Handle any errors during agent creation
        print(f"❌ Error creating agent: {str(e)}")
        return None


# Create an instance of our health advisor agent
health_advisor = create_health_advisor_agent()

## 3. Managing Our Health Conversations 💬
A conversation (or *thread*) is where we'll store the user's messages and the agent's responses about health topics. Let's create a new thread dedicated to Health & Fitness Q&A.


In [ ]:
# A simple data structure to represent a conversation thread
@dataclass
class LocalThread:
    id: str
    messages: List[dict] = field(default_factory=list)


# Function to create a new conversation thread for health discussions
def start_health_conversation():
    """Create a new thread for health & fitness discussions."""
    try:
        # First try to create a real thread via the Azure agents API
        thread = project_client.agents.create_thread()
        print(f"📝 Created a new conversation thread (server-side), ID: {thread.id}")
        return thread
    except Exception as e:
        # If thread creation fails (404 = agents service not available), use local mock
        error_msg = str(e)
        if "404" in error_msg:
            # Create a client-side mock thread instead
            thread = LocalThread(id=f"thread_{uuid.uuid4().hex[:8]}")
            print(f"📝 Created a new conversation thread (client-side), ID: {thread.id}")
            return thread
        else:
            print(f"❌ Error creating thread: {error_msg}")
            return None


# Initialize a new conversation thread that we'll use for our health Q&A
if health_advisor:
    health_thread = start_health_conversation()
else:
    print("⏭️  Skipping thread creation (agent not initialized)")
    health_thread = None

## 4. Asking Health & Fitness Questions 🏃‍♂️
We'll create messages from the user about typical health questions. For example, **"How do I calculate my BMI?"** or **"What's a balanced meal for an active lifestyle?"**. We'll let our Health Advisor Agent respond, always remembering that disclaimer!


In [ ]:
def ask_health_question(thread_id, user_question):
    """Add a user message to the conversation thread.
    
    Args:
        thread_id: ID of the conversation thread
        user_question: The health/fitness question from the user
        
    Returns:
        Message object if successful, None if error occurs
    """
    try:
        # Find the thread object from kernel variables
        thread = None
        
        # Check for fresh thread first (from example queries)
        if 'health_thread_temp' in globals() and thread_id == health_thread_temp.id:
            thread = health_thread_temp
        elif thread_id == health_thread.id:
            thread = health_thread
        
        if thread and isinstance(thread, LocalThread):
            # For mock threads, store message locally
            message = {
                "role": "user",
                "content": user_question
            }
            thread.messages.append(message)
            print(f"📨 Added user question to thread")
            return message
        else:
            # For real threads, use the API
            return project_client.agents.create_message(
                thread_id=thread_id,
                role="user", 
                content=user_question
            )
    except Exception as e:
        print(f"❌ Error adding user message: {e}")
        return None

def process_thread_run(thread_id, agent_id):
    """Ask the agent to process the thread and generate a response using Azure AI Foundry models.
    
    Args:
        thread_id: ID of the conversation thread
        agent_id: ID of the health advisor agent
        
    Returns:
        Run object if successful, None if error occurs
    """
    try:
        # Find the thread object from kernel variables
        thread = None
        
        # Check for fresh thread first (from example queries)
        if 'health_thread_temp' in globals() and thread_id == health_thread_temp.id:
            thread = health_thread_temp
        elif thread_id == health_thread.id:
            thread = health_thread
        
        agent = health_advisor
        
        if thread and isinstance(thread, LocalThread) and isinstance(agent, LocalAgent):
            # For mock threads, use Azure AI Foundry ChatCompletionsClient to generate intelligent response
            latest_msg = thread.messages[-1] if thread.messages else None
            if latest_msg and latest_msg["role"] == "user":
                # Build conversation history with SystemMessage and UserMessage objects
                messages = [
                    SystemMessage(content=agent.instructions)
                ]
                
                # Add all previous messages to provide context
                for msg in thread.messages:
                    if msg["role"] == "user":
                        messages.append(UserMessage(content=msg["content"]))
                    elif msg["role"] == "assistant":
                        messages.append(UserMessage(content=msg["content"]))  # Store assistant response as context
                
                try:
                    # Call Foundry ChatCompletionsClient to generate intelligent response
                    response = chat_client.complete(
                        model=agent.model,
                        messages=messages,
                        temperature=0.7
                    )
                    
                    assistant_response = {
                        "role": "assistant",
                        "content": response.choices[0].message.content
                    }
                    thread.messages.append(assistant_response)
                    print(f"🤖 Run completed with status: completed")
                    return {"status": "completed"}
                    
                except Exception as e:
                    error_msg = str(e)
                    print(f"❌ Model API error: {error_msg}")
                    # Fallback to generic response if API fails
                    assistant_response = {
                        "role": "assistant",
                        "content": (
                            "I understand your question about health and fitness. "
                            "Please remember that I'm not a doctor, and you should always "
                            "consult with healthcare professionals for medical advice. "
                            "That said, I'm happy to provide general wellness information!"
                        )
                    }
                    thread.messages.append(assistant_response)
                    print(f"🤖 Run completed with status: completed (using fallback)")
                    return {"status": "completed"}
            return None
        else:
            # For real threads, use the API
            run = project_client.agents.create_run(
                thread_id=thread_id,
                agent_id=agent_id
            )

            # Poll the run status until completion or error
            while run.status in ["queued", "in_progress", "requires_action"]:
                time.sleep(1)
                run = project_client.agents.get_run(
                    thread_id=thread_id,
                    run_id=run.id
                )

            print(f"🤖 Run completed with status: {run.status}")
            return run
    except Exception as e:
        print(f"❌ Error processing thread run: {str(e)}")
        return None

def view_thread_messages(thread_id):
    """Display all messages in the conversation thread in chronological order.
    
    Args:
        thread_id: ID of the conversation thread to display
    """
    try:
        # Find the thread object from kernel variables
        thread = None
        
        # Check for fresh thread first (from example queries)
        if 'health_thread_temp' in globals() and thread_id == health_thread_temp.id:
            thread = health_thread_temp
        elif thread_id == health_thread.id:
            thread = health_thread
        
        if thread and isinstance(thread, LocalThread):
            # For mock threads, display local messages
            print("🗣️ Conversation so far (oldest to newest):")
            for msg in thread.messages:
                print(f"{msg['role'].upper()}: {msg['content']}\n")
            print("-----------------------------------\n")
        else:
            # For real threads, use the API
            messages = project_client.agents.list_messages(thread_id=thread_id)
            print("\n🗣️ Conversation so far (oldest to newest):")
            
            # Loop through messages in reverse order to show oldest first
            for m in reversed(messages.data):
                if m.content:
                    last_content = m.content[-1]
                    if isinstance(last_content, MessageTextContent):
                        print(f"{m.role.upper()}: {last_content.text.value}\n")
            print("-----------------------------------\n")
    except Exception as e:
        print(f"❌ Error viewing thread: {str(e)}")

### Example Queries
Let's do some quick queries now to see the agent's disclaimers and how it handles typical health questions. We'll ask about **BMI** and about **balanced meal** for an active lifestyle.


In [ ]:
# Create a fresh thread for the example queries (clean results on each run)
if health_advisor:
    print("\n🆕 Fresh Health & Fitness Recommendations\n" + "="*50 + "\n")
    
    # Create fresh thread using LocalThread class
    fresh_thread = LocalThread(id=f"thread_{uuid.uuid4().hex[:8]}")
    
    # Store it temporarily so helper functions can find it
    global health_thread_temp
    health_thread_temp = fresh_thread
    
    # 1) BMI question
    msg1 = ask_health_question(fresh_thread.id, "How do I calculate my BMI, and what does it mean?")
    run1 = process_thread_run(fresh_thread.id, health_advisor.id)

    # 2) Meal plan question  
    msg2 = ask_health_question(fresh_thread.id, "Can you give me a balanced meal plan for someone who exercises 3x a week?")
    run2 = process_thread_run(fresh_thread.id, health_advisor.id)

    # Display the conversation
    print("\n" + "="*50)
    view_thread_messages(fresh_thread.id)
else:
    print("❌ Could not run example queries because agent is None.")

## 5. Cleanup 🧹
If you'd like to remove your agent from the service once finished, you can do so below. (In production, you might keep your agent around for stateful experiences!)

In [ ]:
# Function to clean up and delete the agent when we're done
def cleanup(agent):
    # Only attempt cleanup if we have a valid agent
    if agent:
        try:
            # Check if this is a real server-side agent or a local mock
            if isinstance(agent, LocalAgent):
                # Local mock agent - just remove from memory
                print(f"🗑️ Cleaned up local health advisor agent: {agent.name}")
            else:
                # Real server-side agent - attempt to delete via API
                project_client.agents.delete_agent(agent.id)
                print(f"🗑️ Deleted health advisor agent: {agent.name}")
        except Exception as e:
            # Handle any errors that occur during deletion
            print(f"Error cleaning up agent: {e}")
    else:
        # If no agent was provided, inform the user
        print("No agent to clean up.")

# Call cleanup function to delete our health advisor agent
cleanup(health_advisor)

# Congratulations! 🏆
You've successfully built a **Fun & Fit Health Advisor** that can:
1. **Respond** to basic health and fitness questions.
2. **Use disclaimers** to encourage safe, professional consultation.
3. **Provide** general diet and wellness information.
4. **Use** the synergy of **Azure AI Foundry** modules to power the conversation.

## Next Steps
- Explore adding more advanced tools (like **FileSearchTool** or **CodeInterpreterTool**) to provide more specialized info.
- Evaluate your AI's performance with **azure-ai-evaluation**!
- Add **OpenTelemetry** or Azure Monitor for deeper insights.
- Incorporate **function calling** if you want to handle things like advanced calculation or direct data analysis.

#### Let's proceed to [2-code_interpreter.ipynb](2-code_interpreter.ipynb)

Happy (healthy) coding! 💪